# 07 | Chronological Evaluation Splits & Backtesting Cutoffs
**NHS A&E Intelligent Decision Support System**

**Objective:** inspect the fixed chronological target-period partitions (`train`, `validation`, `test`, `future_unlabelled`) and verify the publication-aware expanding-window training cutoffs (`target_period <= T - 2`) implemented in `src/models/evaluation_splits.py`.

No scalers, imputers, or forecasting models are fitted in this notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs/model_input_manifest.json").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open the project repository before running this notebook.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.evaluation_splits import (
    assign_chronological_splits,
    get_expanding_window_split,
    load_verified_features,
    summarise_splits,
)

feat_df, provenance = load_verified_features(project_root)
split_df = assign_chronological_splits(feat_df)
summary = summarise_splits(split_df)

print("M06 run ID:", provenance["m06_run_id"])
print("M06 SHA-256:", provenance["m06_csv_sha256"])
print("Feature CSV SHA-256:", provenance["feature_csv_sha256"])

part_table = pd.DataFrame.from_dict(summary["partitions"], orient="index")
display(part_table[["target_start", "target_end", "target_months", "total_eligible_rows", "evaluable_rows", "excluded_rows", "distinct_providers_evaluable", "complete_all_features_evaluable"]])


## 1. Publication-Aware Expanding-Window Schedule
Every monthly forecast for target month `T` (decided on the last day of `T - 1`) trains strictly on published target labels `<= T - 2`, excluding unpublished `T - 1` labels.

In [ ]:
schedule = pd.DataFrame(summary["expanding_window_schedule"])
display(schedule)


## 2. Worked Cutoffs: January 2025 (Validation) & January 2026 (Test)

In [ ]:
for tm in ("2025-01-01", "2026-01-01"):
    step = get_expanding_window_split(split_df, tm)
    step_comp = get_expanding_window_split(split_df, tm, complete_features_only=True)
    print(
        f"Target {step['target_period'].strftime('%Y-%m')} ({step['partition']}) | "
        f"Decision {step['decision_date'].strftime('%Y-%m-%d')} | "
        f"Cutoff <= {step['latest_published_period'].strftime('%Y-%m')} | "
        f"Excluded unpublished {step['unpublished_prior_period'].strftime('%Y-%m')}: {step['excluded_unpublished_t_minus_1_rows']} rows | "
        f"Train rows: {len(step['train_df'])} (complete: {len(step_comp['train_df'])}) | "
        f"Eval evaluable: {len(step['eval_evaluable_df'])}, excluded: {len(step['eval_excluded_df'])}"
    )
